# Round 11: tune the target encoding - smoothing strength and column subset - S6E10

Round 10's Exp 31 (counts + leak-free cross-fitted smoothed target encoding + stochastic 3-seed
average) became the new preferred candidate: mean OOF 0.95986, +0.00053 vs the Exp 29 bar with all
five fold deltas positive. Target encoding is now the most productive feature direction. Two
bounded sensitivity tests on the Exp 31 winner, one change each:

- **Exp 33: smoothing strength.** Exp 31 used m=100, picked a priori. Two variants, m=20 (weak
  shrinkage - trusts rare values more) and m=500 (strong shrinkage - rare values pulled hard to
  the fold mean). If the win survives both, it is not an artifact of one lucky m; if one is
  clearly better, the refit takes it. Bounded to these three m values.
- **Exp 34: TE restricted to the continuous columns.** Exp 31 encodes all 21 raw columns, but the
  4 native categoricals already enter the booster as categorical - TE on them may be redundant.
  This variant encodes only Age, Flight Distance, Departure Delay, Arrival Delay. Tests where the
  TE increment actually comes from; bounded to this one subset split.

Construction, bar, and reading rules unchanged from round 10: split-first leakage-safe TE
(early-stop split before any encoding, fit rows cross-fitted over label-blind inner folds with
per-inner-fold smoothing mean, early-stop rows from fit-only tables, outer validation from full
outer-train tables), m=100 except where Exp 33 says otherwise. The bar is now Exp 31's round-10
fold constants, paired per-fold deltas, 2x-spread band as a decision heuristic (never proof). The
incumbent is recomputed in-run as the determinism reference. Refit fires only on a working-signal
win over the Exp 31 bar. The 0.960 target is a mean-OOF target - single folds above it are not
attainment.

In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
CONTINUOUS_TE = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E22_FOLDS_R6 = [0.95886, 0.95756, 0.95953, 0.95875, 0.95922]        # stochastic averaging, round 6
E29_FOLDS_R9 = [0.95938, 0.95804, 0.96033, 0.95933, 0.95956]        # counts + stochastic, round 9
E31_FOLDS_R10 = [0.95998, 0.95873, 0.96056, 0.96000, 0.96000]       # counts + TE, the bar
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
TE_SMOOTH = 100  # Exp 31's smoothing strength; Exp 33 varies it

In [ ]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file() and Path('playground-series-s6e10/data/train.csv').is_file():
    competition_dir = Path('playground-series-s6e10')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])
COUNT_COLS = list(features.columns)
TE_COLS_ALL = COUNT_COLS
missing_cont = [c for c in CONTINUOUS_TE if c not in features.columns]
assert not missing_cont, f'continuous TE columns missing: {missing_cont}'

def to_lgbm(df):
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def add_counts(train_part, other_parts=()):
    tables = {c: train_part[c].value_counts() for c in COUNT_COLS}
    def prep(df):
        out = df.copy()
        for c in COUNT_COLS:
            out[c + '_cnt'] = out[c].map(tables[c]).fillna(0).astype(np.int64)
        return out
    return prep(train_part), [prep(p) for p in other_parts]

def te_table(values, labels, m, gm):
    # smoothed value-level target mean: (sum + m*global) / (count + m)
    stats = labels.groupby(values).agg(['sum', 'count'])
    return (stats['sum'] + m * gm) / (stats['count'] + m)

def te_crossfit(X, y, cols, m=TE_SMOOTH, n_inner=5):
    # cross-fitted encodings for the rows of X themselves: a row's own label never enters its
    # encoding - not through the value table and not through the smoothing mean (recomputed per
    # inner fold from itr rows only). Inner folds are label-blind (plain shuffled KFold).
    te = pd.DataFrame(index=X.index)
    inner = KFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    splits = list(inner.split(np.zeros(len(X))))
    for col in cols:
        enc = np.empty(len(X))
        for itr, iva in splits:
            gm = float(y.iloc[itr].mean())
            t = te_table(X[col].iloc[itr], y.iloc[itr], m, gm)
            enc[iva] = X[col].iloc[iva].map(t).fillna(gm).to_numpy()
        te[col + '_te'] = enc
    return te

def te_apply(X_ref, y_ref, X_new, cols, m=TE_SMOOTH):
    # encodings for rows of X_new from tables built on (X_ref, y_ref) only
    gm = float(y_ref.mean())
    te = pd.DataFrame(index=X_new.index)
    for col in cols:
        t = te_table(X_ref[col], y_ref, m, gm)
        te[col + '_te'] = X_new[col].map(t).fillna(gm).to_numpy()
    return te

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, count_features=False,
             target_encode=False, te_m=TE_SMOOTH, te_cols=None, **params):
    oof = np.zeros(len(feat_frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(feat_frame, label), 1):
        X_tr, y_tr = feat_frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = feat_frame.iloc[va_idx], label.iloc[va_idx]
        if count_features:
            X_tr, (X_va,) = add_counts(X_tr, [X_va])
        # early-stop split happens BEFORE any target encoding, so no early-stop label can
        # touch any early-stop encoding (round-10 audit fix)
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        if target_encode:
            cols = te_cols if te_cols is not None else TE_COLS_ALL
            te_fit = te_crossfit(X_fit, y_fit, cols, m=te_m)
            te_es = te_apply(X_fit, y_fit, X_es, cols, m=te_m)
            te_va = te_apply(X_tr, y_tr, X_va, cols, m=te_m)
            X_fit = pd.concat([X_fit.reset_index(drop=True), te_fit.reset_index(drop=True)], axis=1)
            X_es = pd.concat([X_es.reset_index(drop=True), te_es.reset_index(drop=True)], axis=1)
            X_va = pd.concat([X_va.reset_index(drop=True), te_va.reset_index(drop=True)], axis=1)
        X_fit, X_es, X_va = to_lgbm(X_fit), to_lgbm(X_es), to_lgbm(X_va)
        # caller params merge over defaults (round-6 disclosed fix - never pass a key twice)
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

def fold_aucs(oof, label):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return [roc_auc_score(label.iloc[va], oof[va]) for _, va in skf.split(oof.reshape(-1, 1), label)]

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features')

## Reference: incumbent recompute

Same-run incumbent with the determinism check, as every round.

In [ ]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc

## Exp 33: smoothing strength m = 20 and m = 500

The Exp 31 scheme exactly (counts + TE on all 21 columns + stochastic 3-seed average), with only
the smoothing strength changed. m=20 weakens shrinkage toward the inner-fold mean; m=500
strengthens it. Both variants paired against the Exp 31 bar.

In [ ]:
print('Exp 33: smoothing sensitivity on the Exp 31 scheme')
oof33, aucs33, iters33 = {}, {}, {}
for tag, m in [('33a', 20), ('33b', 500)]:
    oof_seeds = {}
    it = None
    for seed in [42, 43, 44]:
        o, a, i_ = oof_lgbm(features, target, f'E{tag}-s{seed}', seed=seed,
                            count_features=True, target_encode=True, te_m=m, **STOCH)
        oof_seeds[seed] = o
        if seed == 42:
            it = i_
        print(f'  {tag} seed {seed} mean AUC: {np.mean(a):.5f}')
    oof33[tag] = np.mean([oof_seeds[s] for s in [42, 43, 44]], axis=0)
    aucs33[tag] = fold_aucs(oof33[tag], target)
    iters33[tag] = it
    print(f'Exp {tag} (m={m}, averaged) fold AUCs: {[round(a, 5) for a in aucs33[tag]]}')
    print(f'Exp {tag} mean OOF AUC: {np.mean(aucs33[tag]):.5f} +/- {np.std(aucs33[tag]):.5f} | '
          f'pooled: {roc_auc_score(target, oof33[tag]):.6f}')

## Exp 34: TE restricted to the continuous columns

Same Exp 31 scheme, but target encoding only Age, Flight Distance, Departure Delay, Arrival
Delay. The 4 native categoricals keep their native categorical path only. Tests whether the
continuous columns carry the TE increment; bounded to this one subset split.

In [ ]:
print('Exp 34: counts + TE on continuous columns only + stochastic, seeds 42/43/44, averaged')
oof_seeds34 = {}
iters34 = None
for seed in [42, 43, 44]:
    o, a, it = oof_lgbm(features, target, f'E34-s{seed}', seed=seed,
                        count_features=True, target_encode=True, te_cols=CONTINUOUS_TE, **STOCH)
    oof_seeds34[seed] = o
    if seed == 42:
        iters34 = it
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')
oof34 = np.mean([oof_seeds34[s] for s in [42, 43, 44]], axis=0)
aucs34 = fold_aucs(oof34, target)
print(f'Exp 34 (averaged) fold AUCs: {[round(a, 5) for a in aucs34]}')
print(f'Exp 34 mean OOF AUC: {np.mean(aucs34):.5f} +/- {np.std(aucs34):.5f} | pooled: {roc_auc_score(target, oof34):.6f}')

## Results, verdicts, and refit guard

Each variant pairs against the in-run incumbent, the Exp 29 folds, and the Exp 31 folds (the
bar). Refit fires only on a WORKING SIGNAL better vs the Exp 31 bar; on a tie or loss, Exp 31 and
submission_round10.csv stand. Null results are bounded to the tested variant.

In [ ]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return mean_d, spread, label

results = {}
for tag, aucs, desc in [('33a', aucs33['33a'], 'Exp 33a: m=20'),
                        ('33b', aucs33['33b'], 'Exp 33b: m=500'),
                        ('34', aucs34, 'Exp 34: TE continuous-only')]:
    print(desc)
    m_, s_, lab = paired_verdict(aucs, INC_FOLDS)
    print(f'  vs incumbent (in-run):         mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_, s_, lab = paired_verdict(aucs, E29_FOLDS_R9)
    print(f'  vs Exp 29 folds (round 9):     mean delta {m_:+.5f} (spread {s_:.5f}) - {lab}')
    m_bar, s_bar, lab_bar = paired_verdict(aucs, E31_FOLDS_R10)
    print(f'  vs Exp 31 folds (round 10 bar): mean delta {m_bar:+.5f} (spread {s_bar:.5f}) - {lab_bar}')
    results[tag] = (m_bar, lab_bar, float(np.mean(aucs)))

winners = [k for k, (m, lab, _) in results.items() if lab.startswith('WORKING SIGNAL: better')]
win = max(winners, key=lambda k: results[k][2]) if winners else None
print()
if win:
    print(f'REFIT GUARD: Exp {win} clears the Exp 31 bar under the paired band -> refit fires (exploratory, descriptive signal).')
else:
    print('REFIT GUARD: nothing cleared the Exp 31 bar under the paired band. Exp 31 remains the '
          'preferred candidate; submission_round10.csv is untouched.')

In [ ]:
if win is None:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    if win in ('33a', '33b'):
        m_win = 20 if win == '33a' else 500
        cols_win, it_win = TE_COLS_ALL, iters33[win]
        scheme = f'Exp {win}: counts + cross-fitted TE (m={m_win}) + stochastic, 3 seeds'
    else:  # win == '34'
        m_win, cols_win, it_win = TE_SMOOTH, CONTINUOUS_TE, iters34
        scheme = 'Exp 34: counts + cross-fitted TE (continuous only) + stochastic, 3 seeds'
    basis = int(round(np.mean(it_win)))
    tr_c, (te_c,) = add_counts(features, [test_features])
    te_full = te_crossfit(tr_c, target, cols_win, m=m_win)   # full-train cross-fitted
    te_test = te_apply(features, target, test_features, cols_win, m=m_win)
    X_full = pd.concat([tr_c.reset_index(drop=True), te_full.reset_index(drop=True)], axis=1)
    X_test = pd.concat([te_c.reset_index(drop=True), te_test.reset_index(drop=True)], axis=1)
    X_full, X_test = to_lgbm(X_full), to_lgbm(X_test)
    preds = []
    for seed in [42, 43, 44]:
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=basis,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(STOCH)
        model = lgb.LGBMClassifier(**cfg)
        model.fit(X_full, target)
        preds.append(model.predict_proba(X_test)[:, 1])
    sub = pd.DataFrame({'id': test['id'], 'satisfaction': np.mean(preds, axis=0)})
    sub.to_csv('submission_round11.csv', index=False)
    print(f'submission_round11.csv written: {len(sub):,} rows | {scheme} | n_estimators basis {basis} '
          f'(in-run seed-42 fold-mean best_iteration). Local file only - NOT submitted; Andrea submits.')

In [ ]:
# OOF cache (established pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round11_oof.npz',
         incumbent=oof_inc, exp33a=oof33['33a'], exp33b=oof33['33b'], exp34=oof34,
         target=target.to_numpy())
meta = {
    'round': 11,
    'train_sha256': h.hexdigest(),
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'configs': {
        'incumbent': {'note': 'Exp 8 config, in-run reference'},
        'exp33a': {'scheme': 'Exp 31 scheme with TE m=20'},
        'exp33b': {'scheme': 'Exp 31 scheme with TE m=500'},
        'exp34': {'scheme': 'Exp 31 scheme, TE on continuous columns only'},
        'stoch': STOCH,
    },
    'fold_aucs': {'incumbent': INC_FOLDS, 'exp33a': aucs33['33a'],
                  'exp33b': aucs33['33b'], 'exp34': aucs34},
}
(cache_dir / 'round11_oof.json').write_text(_json.dumps(meta, indent=2))
print('cached oof_cache/round11_oof.npz + round11_oof.json')

## What happens next

- If a smoothing variant beats the bar: the TE increment is not an artifact of m=100 - the refit
  takes the stronger m, and a finer sweep around it is bounded and cheap.
- If both m variants tie the bar: the win is insensitive to smoothing strength across 20-500 in
  this test - m=100 stands without further tuning.
- If Exp 34 beats or ties the bar: the continuous columns carry the TE increment in this test -
  dropping the redundant categorical TE simplifies the winner (ties favor the simpler model).
- If Exp 34 loses clearly: the categorical TE earns its place here - keep all-21 TE and the next
  direction is count/TE crosses or the pending external-data question (needs Andrea's explicit
  OK first).